# Power BI Migration Optimizer — Score Enrichment Notebook

Run this notebook in a **Microsoft Fabric workspace** to:
1. Discover all Power BI semantic models in a workspace
2. Run Best Practice Analyzer (BPA) via Semantic Link Labs
3. Pull model memory statistics from XMLA
4. POST the enriched metrics to the Migration Optimizer platform

**Prerequisites:**
- `semantic-link` and `semantic-link-labs` installed (see cell below)
- The Migration Optimizer platform running at `OPTIMIZER_URL`
- Fabric workspace permissions: at least **Contributor** on the datasets you want to audit

**Instructions:**
1. Set `WORKSPACE_NAME` and `OPTIMIZER_URL` in the Configuration cell
2. Run all cells
3. Open the Optimizer dashboard to see enriched scores

In [ ]:
# Install required packages (run once)
%pip install semantic-link semantic-link-labs --quiet

In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────
# Set these before running

WORKSPACE_NAME = "Finance Analytics"          # Power BI workspace to audit
OPTIMIZER_URL  = "http://localhost:8000"       # URL of the Migration Optimizer
DRY_RUN        = False                         # Set True to print payloads without posting

# The optimizer requires an API key (X-API-Key header). Don't paste it here – read it
# from a secret store, e.g. Azure Key Vault via notebookutils:
#   OPTIMIZER_API_KEY = notebookutils.credentials.getSecret("https://<vault>.vault.azure.net/", "optimizer-api-key")
import os
OPTIMIZER_API_KEY = os.environ.get("OPTIMIZER_API_KEY", "")
HEADERS = {"X-API-Key": OPTIMIZER_API_KEY}

print(f"Workspace : {WORKSPACE_NAME}")
print(f"Optimizer : {OPTIMIZER_URL}")
print(f"Dry run   : {DRY_RUN}")
print(f"API key   : {'set' if OPTIMIZER_API_KEY else 'NOT SET – posts will be rejected (401)'}")

In [ ]:
import requests
import sempy.fabric as spf
import pandas as pd

try:
    import sempy_labs as labs
    HAS_LABS = True
    print("semantic-link-labs available — BPA will run")
except ImportError:
    HAS_LABS = False
    print("semantic-link-labs not available — BPA skipped (model health scores will use defaults)")

In [ ]:
# ── STEP 1: Discover datasets ──────────────────────────────────────────────────
datasets_df = spf.list_datasets(workspace=WORKSPACE_NAME)
print(f"Found {len(datasets_df)} datasets in '{WORKSPACE_NAME}'")
datasets_df[['Dataset Id', 'Dataset Name', 'Configured By']]

In [ ]:
# ── STEP 2: Audit each dataset ────────────────────────────────────────────────
# Each function returns ONLY the metrics it actually measured. If a step fails, its
# keys are left out, and the optimizer shows that dimension as "not measured".
# (Sending zeros instead would be read as "no issues" – e.g. a 0 GB, perfect model.)

def run_bpa(dataset_name: str, workspace: str) -> dict:
    """Run Best Practice Analyzer and return violation counts."""
    if not HAS_LABS:
        return {}
    try:
        bpa_df = labs.run_model_bpa(dataset=dataset_name, workspace=workspace, return_dataframe=True)
        critical = int(len(bpa_df[bpa_df['Severity'] == 'Error']))
        warnings = int(len(bpa_df[bpa_df['Severity'] == 'Warning']))
        return {"bpa_critical_count": critical, "bpa_warning_count": warnings}
    except Exception as e:
        print(f"  BPA not measured for {dataset_name}: {e}")
        return {}


def _col(df, *names):
    """First matching column name (Semantic Link column names vary between versions)."""
    for n in names:
        if n in df.columns:
            return n
    raise KeyError(f"none of {names} in {list(df.columns)}")


def get_model_metrics(dataset_name: str, workspace: str) -> dict:
    """Model size, table/column counts and high-cardinality columns (>1M distinct values)."""
    try:
        tables = spf.list_tables(dataset=dataset_name, workspace=workspace, extended=True)
        columns = spf.list_columns(dataset=dataset_name, workspace=workspace, extended=True)
        metrics = {
            "table_count": int(len(tables)),
            "column_count": int(len(columns)),
        }
        size_col = _col(tables, "Total Size", "Size")
        metrics["model_size_gb"] = round(float(tables[size_col].sum()) / 1e9, 2)
        card_col = _col(columns, "Cardinality", "Column Cardinality")
        metrics["high_cardinality_columns"] = int((columns[card_col] > 1_000_000).sum())
        return metrics
    except Exception as e:
        print(f"  Model size/complexity not measured for {dataset_name}: {e}")
        return {}


def get_relationship_metrics(dataset_name: str, workspace: str) -> dict:
    """Count bidirectional and many-to-many relationships."""
    try:
        rel_df = spf.list_relationships(dataset=dataset_name, workspace=workspace)
        cross = rel_df[_col(rel_df, "Cross Filtering Behavior")].astype(str).str.lower()
        multiplicity = rel_df[_col(rel_df, "Multiplicity")].astype(str).str.lower()
        return {
            "bidirectional_rel_count": int(cross.str.contains("both").sum()),
            "many_to_many_rel_count": int((multiplicity == "m:m").sum()),
        }
    except Exception as e:
        print(f"  Relationships not measured for {dataset_name}: {e}")
        return {}


print("Audit functions defined. Ready to run.")

In [ ]:
# ── STEP 3: Run audit and post results ────────────────────────────────────────

results = []

for _, row in datasets_df.iterrows():
    dataset_id   = row['Dataset Id']
    dataset_name = row['Dataset Name']
    print(f"\nAuditing: {dataset_name}")

    # Collect all metrics (only what was actually measured)
    payload = {"dataset_id": dataset_id}
    payload.update(run_bpa(dataset_name, WORKSPACE_NAME))
    payload.update(get_model_metrics(dataset_name, WORKSPACE_NAME))
    payload.update(get_relationship_metrics(dataset_name, WORKSPACE_NAME))
    measured = sorted(k for k in payload if k != "dataset_id")
    print(f"  Measured: {', '.join(measured) or 'nothing'}")

    if len(payload) == 1:
        status = "skipped — nothing measured"
    elif DRY_RUN:
        print(f"  [DRY RUN] Would POST: {payload}")
        status = "dry-run"
    else:
        try:
            r = requests.post(
                f"{OPTIMIZER_URL}/api/datasets/{dataset_id}/bpa-update",
                json=payload,
                headers=HEADERS,
                timeout=10,
            )
            status = f"HTTP {r.status_code}"
            if r.status_code == 401:
                status = "401 — missing or wrong OPTIMIZER_API_KEY"
            elif r.status_code == 404:
                status = "404 — dataset not in optimizer (connect Power BI first)"
        except requests.exceptions.ConnectionError:
            status = "Connection error — is the optimizer running?"

    results.append({"Dataset": dataset_name, "Dataset Id": dataset_id, "Status": status})

print("\n─── Audit Complete ───")
pd.DataFrame(results)

In [ ]:
# ── STEP 4: Fetch updated scores from the optimizer ───────────────────────────
try:
    r = requests.get(f"{OPTIMIZER_URL}/api/datasets", headers=HEADERS, timeout=10)
    r.raise_for_status()
    scores = r.json()
    score_df = pd.DataFrame([{
        'Dataset'    : d['dataset_name'],
        'Workspace'  : d['workspace_name'],
        'Score'      : d['total_score'],
        'Decision'   : d['decision'],
        'Risk'       : d['migration_risk'],
        'Effort (d)' : d['estimated_effort_days'],
    } for d in scores])
    print(score_df.to_string(index=False))
except Exception as e:
    print(f"Could not fetch scores: {e}")